In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    print(f"🔁 Using token #{token_index + 1}")
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_keyword_check_output.csv"

df = pd.read_csv(input_path)

# === Init updated columns ===
df["android_metadata_match"] = df["android_metadata_match"].astype(str)
df["android_in_readme"] = df["android_in_readme"].astype(str)

def check_android_in_readme_like(repo):
    # 1. Default README
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(readme_url, headers=get_headers())
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
            if "android" in decoded:
                return True
        except:
            pass

    # 2. .md/.rst fallback
    contents_url = f"https://api.github.com/repos/{repo}/contents"
    r = requests.get(contents_url, headers=get_headers())
    if r.status_code == 200:
        for item in r.json():
            name = item.get("name", "").lower()
            if name.endswith((".md", ".markdown", ".rst")):
                file_url = item.get("download_url")
                if file_url:
                    try:
                        text = requests.get(file_url, headers=get_headers()).text.lower()
                        if "android" in text:
                            return True
                    except:
                        pass

    # 3. Wiki fallback
    wiki_url = f"https://raw.githubusercontent.com/wiki/{repo}/Home.md"
    r = requests.get(wiki_url, headers={"User-Agent": "android-repo-crawler/1.0"})
    if r.status_code == 200 and "android" in r.text.lower():
        return True

    return False

# === Main loop ===
for i, row in df.iterrows():
    if row["Valid_Repo"] != "yes":
        continue

    repo = row["full_name"]

    # Step 1: Fetch latest metadata
    meta_url = f"https://api.github.com/repos/{repo}"
    r = requests.get(meta_url, headers=get_headers())
    name, desc = row.get("name", ""), ""
    topics = row.get("topics", "")
    if r.status_code == 200:
        data = r.json()
        name = str(data.get("name", "")).lower()
        desc = str(data.get("description", "")).lower()
        topics_url = f"https://api.github.com/repos/{repo}/topics"
        r2 = requests.get(topics_url, headers=get_headers())
        if r2.status_code == 200:
            topic_names = r2.json().get("names", [])
            topics = ",".join(topic_names).lower()

    metadata_hit = "android" in name or "android" in desc or "android" in topics
    df.at[i, "android_metadata_match"] = "yes" if metadata_hit else "no"

    # Step 2: Re-check README or wiki
    in_readme = check_android_in_readme_like(repo)
    df.at[i, "android_in_readme"] = "yes" if in_readme else "no"

    if i % 100 == 0:
        print(f"🔎 Checked {i+1} repos...")

# === Save output ===
df.to_csv(output_path, index=False)
print(f"✅ Step 4 complete. Saved to: {output_path}")
